# Processing data

## 1. Data tieng Viet

In [ ]:
import pandas as pd
import re
import unicodedata
from underthesea import word_tokenize

VI_STOP = set("""
và là của cho các có một những này đó ở trong khi với đến từ về thì
được không hơn rất cũng lại đã đang sẽ hay vì do tại theo như
""".split())

def normalize_vi(text: str) -> str:
    """Chuẩn hoá tiếng Việt: unicode NFC, lowercase, remove url/email, giữ chữ + khoảng trắng."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""

    text = str(text)

    text = unicodedata.normalize("NFC", text)

    text = text.lower()

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    text = re.sub(
        r"[^a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩ"
        r"òóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ\s]",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_vi(text: str):
    """Tách từ tiếng Việt bằng underthesea."""
    if not text:
        return []
    return word_tokenize(text, format="text").split()

def remove_stopwords_vi(tokens):
    return [t for t in tokens if t not in VI_STOP and len(t) > 1]

def preprocess_vi(text: str):
    norm = normalize_vi(text)
    tokens = tokenize_vi(norm)
    tokens = remove_stopwords_vi(tokens)
    final_text = " ".join(tokens)
    return norm, tokens, final_text

input_csv = "data_1.csv"
output_csv = "output_vi_preprocessed.csv"
col = "Thông tin"   

df = pd.read_csv(input_csv)

res = df[col].apply(preprocess_vi)
df["thong_tin_norm_vi"] = res.apply(lambda x: x[0])
df["tokens_vi"] = res.apply(lambda x: x[1])
df["thong_tin_final_vi"] = res.apply(lambda x: x[2])

df.to_csv(output_csv, index=False, encoding="utf-8-sig")
print("✅ Saved:", output_csv)


✅ Saved: output_vi_preprocessed.csv


## Data tieng Anh

In [ ]:
import nltk

packages = [
    "punkt",
    "punkt_tab", 
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng", 
]

for p in packages:
    try:
        nltk.data.find(p)
    except:
        nltk.download(p)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nl

In [ ]:
import pandas as pd
import re
import unicodedata
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

EN_STOP = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def normalize_en(text: str) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.lower()

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    text = re.sub(r"[^a-z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_en(text: str):
    norm = normalize_en(text)

    tokens = word_tokenize(norm)

    tokens = [t for t in tokens if t not in EN_STOP and len(t) > 2]

    pos_tags = nltk.pos_tag(tokens)
    lemmas = [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        for word, pos in pos_tags
    ]

    final_text = " ".join(lemmas)
    return norm, lemmas, final_text


In [10]:
input_csv = "data_translation.csv"
output_csv = "output_en_processed.csv"

df = pd.read_csv(input_csv)

res = df["information"].apply(preprocess_en)

df["thong_tin_norm_en"] = res.apply(lambda x: x[0])
df["tokens_en"] = res.apply(lambda x: x[1])
df["thong_tin_final_en"] = res.apply(lambda x: x[2])

df.to_csv(output_csv, index=False, encoding="utf-8-sig")
print("✅ EN processing done:", output_csv)


✅ EN processing done: output_en_processed.csv


## Merge


In [ ]:
import pandas as pd

vi_file = "output_vi_processed.csv"
en_file = "output_en_processed.csv"
out_file = "output_augmented_vi_en.csv"

df_vi = pd.read_csv(vi_file)
df_en = pd.read_csv(en_file)

df_vi_aug = df_vi[[
    "STT", "Thông tin", "Label",
    "thong_tin_norm_vi", "tokens_vi", "thong_tin_final_vi"
]].copy()

df_vi_aug = df_vi_aug.rename(columns={
    "thong_tin_norm_vi": "thong_tin_norm",
    "tokens_vi": "tokens",
    "thong_tin_final_vi": "thong_tin_final"
})
df_vi_aug["language"] = "vi"

df_en_aug = df_en[[
    "STT", "Thông tin", "Label",
    "thong_tin_norm_en", "tokens_en", "thong_tin_final_en"
]].copy()

df_en_aug = df_en_aug.rename(columns={
    "thong_tin_norm_en": "thong_tin_norm",
    "tokens_en": "tokens",
    "thong_tin_final_en": "thong_tin_final"
})
df_en_aug["language"] = "en"

df_merged = pd.concat([df_vi_aug, df_en_aug], axis=0, ignore_index=True)

df_merged["STT"] = range(1, len(df_merged) + 1)

df_merged.to_csv(out_file, index=False, encoding="utf-8-sig")
print("✅ Saved:", out_file)


✅ Saved: output_augmented_vi_en.csv
